# Fix a colleague's summer sale notebook

*Python Foundations · Topic 11.1*

Your colleague's notebook reports June 2024's summer sale by market: sessions, revenue and revenue per session. Every cell has the output from their session, and the last one says the sale earned $1.59 per US session. Find out what the code on the page really gives, and fix the notebook so that it gives the same result every time it runs from the top.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Download [sessions_export_2024_06.csv](https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_06.csv) and upload it to the `simmer-python` folder in your Drive.
2. Before you run anything, read the execution counts next to your colleague's code cells, from the top. Which cells ran in a different order from the page?
3. Choose **Runtime → Restart session and run all**. Fix each cell that stops with an error, then restart and run all again:
   - If a cell uses a function that's defined further down, move the cell that defines it up, with **Move cell up**.
   - If a cell uses a variable that no cell creates, add the missing cell above it. `market_revenue` is a dictionary with each market's total revenue, as a float, built the same way as `market_sessions`.
4. Run the cell that totals `market_sessions` twice. Fix it so that it gives the same result however many times it runs, and delete any cell you don't need any more.
5. Choose **Runtime → Restart session and run all**, and compare the last cell's output with your colleague's.

**Expected output**: after a restart and run all, `market_sessions` is `{'ca': 2278, 'us': 5195}`, and the last cell prints:

    ca: 2,278 sessions, $4,845.30 revenue, $2.13 per session
    us: 5,195 sessions, $16,493.76 revenue, $3.17 per session

The **Check your work** cell runs last, with the others. It checks the results, and whether every cell in the session ran once, with no errors.

## Summer sale 2024 by market

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import csv

CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_06.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

1933

In [6]:
sale_rows = [row for row in rows if is_sale(row["session_campaign"])]
len(sale_rows)

183

In [5]:
def is_sale(campaign):
    """Return True for the summer sale campaigns."""
    return campaign.endswith("_summersale")

In [4]:
market_sessions = {}

In [9]:
for row in sale_rows:
    market = row["session_campaign"].split("_")[1]
    market_sessions[market] = market_sessions.get(market, 0) + int(row["sessions"])
market_sessions

{'ca': 4556, 'us': 10390}

In [10]:
for market in market_sessions:
    revenue_per_session = market_revenue[market] / market_sessions[market]
    print(f"{market}: {market_sessions[market]:,} sessions, ${market_revenue[market]:,.2f} revenue, ${revenue_per_session:.2f} per session")

ca: 4,556 sessions, $4,845.30 revenue, $1.06 per session
us: 10,390 sessions, $16,493.76 revenue, $1.59 per session


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import re
    import sys
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])

    path = found.get("CSV_PATH")
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Run the cell that reads the file."))
    elif not path.replace("\\", "/").endswith("sessions_export_2024_06.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The exercise uses June's export, sessions_export_2024_06.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}. Check that Drive is mounted and that the file is in the simmer-python folder, then copy its path from the Files panel."))
    else:
        results.append((True, "CSV_PATH points to June's export."))

    rows = found.get("rows")
    if not isinstance(rows, list):
        results.append((False, "rows isn't a list. Run the cell that reads the file."))
    elif len(rows) == 1933:
        results.append((True, "rows has all 1,933 rows of the export."))
    else:
        results.append((False, f"rows has {len(rows)} rows, but sessions_export_2024_06.csv has 1,933."))

    is_sale = found.get("is_sale")
    sale_rows = found.get("sale_rows")
    if not callable(is_sale):
        results.append((False, "is_sale() isn't defined. Run the cell that defines it, above the cell that uses it."))
    elif not isinstance(sale_rows, list):
        results.append((False, "sale_rows isn't a list. Move the cell that defines is_sale() above the cell that creates sale_rows, and restart and run all."))
    elif len(sale_rows) == 183:
        results.append((True, "sale_rows has the 183 summer sale rows."))
    else:
        results.append((False, f"sale_rows has {len(sale_rows)} rows, but June has 183 summer sale rows. Keep your colleague's is_sale() as it is."))

    def adds_from_scratch(name):
        """Whether the last cell that adds to the dictionary also starts it, or None if no cell does."""
        adding = [source for source in history if re.search(rf"\b{name}\[[^\]]+\]\s*\+?=", source)]
        if not adding:
            return None
        return bool(re.search(rf"^{name}\s*=\s*(\{{\}}|dict\(\))", adding[-1], re.M))

    sessions = found.get("market_sessions")
    expected = {"ca": 2278, "us": 5195}
    if not isinstance(sessions, dict):
        results.append((False, "market_sessions isn't a dictionary. Run the cell that creates it."))
    elif sessions == expected:
        results.append((True, "market_sessions has each market's sessions, counted once."))
    elif all(sessions.get(market) == 2 * value for market, value in expected.items()):
        results.append((False, "market_sessions has twice the sessions: the loop ran twice and added the month to its total again. Start the dictionary in the loop's cell, then restart and run all."))
    elif all(isinstance(sessions.get(market), int) and sessions[market] > value and sessions[market] % value == 0 for market, value in expected.items()):
        results.append((False, f"market_sessions is {sessions!r}: the month's sessions added more than once. Start the dictionary in the loop's cell, then restart and run all."))
    else:
        results.append((False, f"market_sessions should be {expected}, but it is {sessions!r}."))
    if history is not None and adds_from_scratch("market_sessions") is False:
        results.append((False, "The cell that adds to market_sessions doesn't start it, so every run adds to the last one. Make market_sessions = {} the first line of that cell, and delete the cell it was in."))

    revenue = found.get("market_revenue")
    expected_revenue = {"ca": 4845.30, "us": 16493.76}
    if revenue is None:
        results.append((False, "market_revenue isn't defined. Add a cell above the last one that creates it: each market's total revenue, as a float, the same way as market_sessions."))
    elif not isinstance(revenue, dict):
        results.append((False, f"market_revenue should be a dictionary, but it is a {type(revenue).__name__}."))
    elif set(revenue) != set(expected_revenue):
        results.append((False, f"market_revenue has the keys {sorted(revenue)!r}. Use the same markets as market_sessions: row[\"session_campaign\"].split(\"_\")[1]."))
    elif any(isinstance(value, str) for value in revenue.values()):
        results.append((False, "market_revenue has text values. Convert each row's revenue with float() before you add it."))
    elif all(abs(revenue[market] - value) < 0.01 for market, value in expected_revenue.items()):
        results.append((True, "market_revenue has each market's revenue."))
    elif all(abs(revenue[market] - 2 * value) < 0.01 for market, value in expected_revenue.items()):
        results.append((False, "market_revenue has twice the revenue: its loop ran twice. Start the dictionary in the loop's cell, then restart and run all."))
    else:
        results.append((False, f"market_revenue should be about {expected_revenue}, but it is {revenue!r}. Add float(row[\"revenue\"]) for every row in sale_rows."))
    if history is not None and adds_from_scratch("market_revenue") is False:
        results.append((False, "The cell that adds to market_revenue doesn't start it, so every run adds to the last one. Make market_revenue = {} the first line of that cell."))

    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. The notebook's outputs now come from the code on the page.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to fix it: if the check passes, your way is right too. The fixed notebook, from the top:

Move the cell that defines `is_sale()` above the cell that creates `sale_rows`. Move `market_sessions = {}` into the loop's cell and delete its old cell. Add the `market_revenue` cell above the last one.

A text cell:

```markdown
## Summer sale 2024 by market
```

```python
from google.colab import drive

drive.mount("/content/drive")
```

```python
import csv

CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_06.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)
```

```python
def is_sale(campaign):
    """Return True for the summer sale campaigns."""
    return campaign.endswith("_summersale")
```

```python
sale_rows = [row for row in rows if is_sale(row["session_campaign"])]
len(sale_rows)
```

```python
market_sessions = {}
for row in sale_rows:
    market = row["session_campaign"].split("_")[1]
    market_sessions[market] = market_sessions.get(market, 0) + int(row["sessions"])
market_sessions
```

```python
market_revenue = {}
for row in sale_rows:
    market = row["session_campaign"].split("_")[1]
    market_revenue[market] = market_revenue.get(market, 0) + float(row["revenue"])
market_revenue
```

```python
for market in market_sessions:
    revenue_per_session = market_revenue[market] / market_sessions[market]
    print(f"{market}: {market_sessions[market]:,} sessions, ${market_revenue[market]:,.2f} revenue, ${revenue_per_session:.2f} per session")
```

</details>